In [0]:
from datetime import datetime
from pyspark.sql import DataFrame
import logging
import pandas as pd
from io import BytesIO

In [0]:
# Configuração do logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

log = logging.getLogger("pipeline_ingestion")

logging.getLogger("azure").setLevel(logging.WARNING)
logging.getLogger("azure.identity").setLevel(logging.WARNING)
logging.getLogger("azure.core").setLevel(logging.WARNING)

In [0]:
def log_ingestion(
    snapshot_id: int,
    arquivo: str,
    registros: int,
    status: str,
    mensagem: str
) -> None:
    """
    Registra informações de processamento de arquivos no log de ingestão
    e também escreve mensagens no log da aplicação.

    Args:
        snapshot_id (int):
            Identificador da execução do polling.

        arquivo (str):
            Caminho completo do arquivo processado.

        registros (int):
            Quantidade de registros processados.

        status (str):
            Status da execução.
            Ex.: "SUCESSO" ou "ERRO".

        mensagem (str):
            Mensagem descritiva do resultado da execução.

    Returns:
        None
    """

    log_entry = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "snapshot_id": snapshot_id,
        "arquivo": arquivo,
        "registros": registros,
        "status": status,
        "mensagem": mensagem
    }

    ingestion_log.append(log_entry)

In [0]:

def read_parquet_to_spark_df(file_path: str) -> DataFrame:
    """
    Realiza o download de um arquivo Parquet armazenado no ADLS,
    converte o conteúdo para um DataFrame Spark e retorna o resultado.

    Args:
        file_path (str): Caminho completo do arquivo no ADLS.

    Returns:
        DataFrame: DataFrame Spark contendo os dados do arquivo.
    """

    data = (
        container_client
        .get_file_client(file_path)
        .download_file()
        .readall()
    )
    
    pdf = pd.read_parquet(BytesIO(data))

    # Converter tudo para string porque alguns arquivos possuem colunas vazias causando problemas na inferência dos tipos das colunas
    pdf = pdf.astype(str)

    return spark.createDataFrame(pdf)

In [0]:
def list_files(
    container_client,
    folder_name: str = None,
    file_name_contains: str = None
) -> list:
    """
    Lista arquivos do container ADLS com filtros opcionais.

    Args:
        container_client:
            Cliente do File System do ADLS.

        folder_name (str, optional):
            Caminho da pasta a ser consultada.
            Se não informado, consulta todo o container.

        file_name_contains (str, optional):
            Trecho que deve existir no nome do arquivo.
            Se não informado, retorna todos os arquivos encontrados.

    Returns:
        list:
            Lista contendo os caminhos completos dos arquivos encontrados.
    """

    files = []

    paths = (
        container_client.get_paths(folder_name)
        if folder_name
        else container_client.get_paths()
    )

    for path in paths:

        if path.is_directory:
            continue

        if file_name_contains and file_name_contains not in path.name:
            continue

        files.append(path.name)

    return files

In [0]:
def write_sql_server(
    df: DataFrame,
    table_name: str,
    jdbc_hostname: str,
    jdbc_database: str,
    jdbc_username: str,
    jdbc_password: str,
    mode: str = "append"
) -> None:
    """
    Escreve um DataFrame Spark em uma tabela do SQL Server.

    Args:
        df (DataFrame):
            DataFrame Spark a ser gravado.

        table_name (str):
            Nome completo da tabela destino
            (ex.: "squad2.ecommerce_clientes").

        jdbc_hostname (str):
            Host do SQL Server.

        jdbc_database (str):
            Nome do banco de dados.

        jdbc_username (str):
            Usuário de acesso ao banco.

        jdbc_password (str):
            Senha de acesso ao banco.

        mode (str, optional):
            Modo de escrita do Spark.
            Ex.: append, overwrite, ignore, errorifexists.
            Default: append.
    """

    df.write \
        .format("sqlserver") \
        .option("host", jdbc_hostname) \
        .option("port", 1433) \
        .option("database", jdbc_database) \
        .option("dbtable", table_name) \
        .option("user", jdbc_username) \
        .option("password", jdbc_password) \
        .mode(mode) \
        .save()